In [ ]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Radiology

In [ ]:
# Load the data
radiology_query = """
SELECT
    es.subject_id,
    es.stay_id,
    es.hadm_id,
    rad.note_id,
    rad.charttime,
    det.field_name,
    det.field_value

FROM physionet-data.mimiciv_ed.edstays es

INNER JOIN physionet-data.mimiciv_note.radiology rad
    ON rad.subject_id = es.subject_id
    AND rad.charttime >= es.intime
    AND rad.charttime <= es.outtime

INNER JOIN physionet-data.mimiciv_note.radiology_detail det
    ON rad.note_id = det.note_id

WHERE det.field_name = 'exam_name'
"""

df_rad = client.query(radiology_query).to_dataframe()

print(df_rad.shape)
print(f"Unique ED stays: {df_rad['stay_id'].nunique()}")
print(f"Unique subjects: {df_rad['subject_id'].nunique()}")

df_rad["charttime"] = pd.to_datetime(df_rad["charttime"], utc=True)

In [ ]:
# Filter to exam_name rows and clean
df_rad["exam_name"] = df_rad["field_value"].str.upper().str.strip()

admin_patterns = [
    "BY SAME PHYSICIAN", "BY DIFFERENT PHYSICIAN",
    "DISTINCT PROCEDURAL SERVICE", "SEPARATE STRUCTURE",
    "MOD SEDATION", "FEE ADJUSTED", "___", "CAD "
]
mask_admin = df_rad["exam_name"].str.contains("|".join(admin_patterns), na=False)
df_rad = df_rad[~mask_admin].copy()
print(f"\nRows after removing admin entries: {len(df_rad)}")

In [ ]:
# Resolve field_value format
# field_value can be one of three formats:
#   a) CPT code       — purely numeric, e.g. "71046"
#   b) Exam code      — short uppercase label, e.g. "CHEST (PORTABLE AP)"
#   c) Literal name   — verbose, e.g. "COMPUTED TOMOGRAPHY CHEST W/CONTRAST"
#
# Each CPT code maps directly to an acuity level (0, 1, or 2)
# rather than routing through modality/region as an intermediate step

cpt_acuity = {
    # Level 2 — high acuity
    "71250": 2, "71260": 2, "71270": 2,   # CT chest
    "71275": 2,                            # CTA chest
    "74177": 2, "74178": 2,               # CT abdomen/pelvis combined
    "70450": 2, "70460": 2, "70470": 2,   # CT head
    "70496": 2, "70498": 2,               # CTA head/neck
    "70551": 2, "70552": 2, "70553": 2,   # MRI brain
    "77001": 2, "77002": 2,               # fluoroscopy/line placement

    # Level 1 — moderate acuity
    "71045": 1, "71046": 1,               # chest X-ray (1-2 views)
    "71047": 1, "71048": 1,               # chest X-ray (3-4 views)
    "74150": 1, "74160": 1, "74170": 1,   # CT abdomen only
    "74179": 1,                            # CT abdomen/pelvis w/o contrast
    "72192": 1, "72193": 1, "72194": 1,   # CT pelvis
    "72141": 1, "72146": 1, "72148": 1,   # MRI spine
    "76700": 1, "76705": 1,               # ultrasound abdomen
    "76856": 1, "76857": 1,               # ultrasound pelvis
    "93306": 1, "93307": 1, "93308": 1,   # echocardiogram
}

def is_cpt_code(value):
    return bool(re.match(r"^\d{5}$", str(value).strip()))

In [ ]:
# Assign acuity level per exam row
# Level 2: high-acuity — cross-sectional imaging of critical regions,
#          vascular studies, bedside/portable imaging (implies unstable patient)
# Level 1: moderate-acuity — standard imaging, non-urgent workup
# Level 0: low/routine — mammography, elective extremity films, other non-urgent

high_acuity_strings = [
    "CTA ",
    "CT HEAD", "CT CHEST", "CT ABD", "CT PELVIS", "CT C-SPINE",
    "COMPUTED TOMOGRAPHY HEAD", "COMPUTED TOMOGRAPHY CHEST",
    "MR HEAD", "MRI BRAIN", "MAGNETIC RESONANCE HEAD",
    "CHEST (PORTABLE", "CHEST PORT", "PORTABLE CHEST",
    "FLUORO", "LINE PLACEMENT", "GUIDANCE",
]

moderate_acuity_strings = [
    "CT ",          # catches remaining CTs not matched above
    " CT",
    "MRI", "MR ",   # catches remaining MRIs
    "ULTRASOUND", " US", "US.",
    "RENAL U", "PELVIS U",
    "ECHO",
    "PA & LAT", "AP & LAT",   # standard chest X-ray views
    "X-RAY", "RADIOGRAPH",
]

def assign_acuity(name):
    if is_cpt_code(name):
        return cpt_acuity.get(name.strip(), 0)   # unknown CPTs → 0
    # Check high acuity first — order matters
    if any(h in name for h in high_acuity_strings):
        return 2
    if any(m in name for m in moderate_acuity_strings):
        return 1
    return 0   # mammography, elective extremity, unrecognized → routine

df_rad["exam_acuity"] = df_rad["exam_name"].apply(assign_acuity)

# Audit unresolved rows (acuity=0 from string matching, not CPT)
unresolved = df_rad[
    (df_rad["exam_acuity"] == 0) & (~df_rad["exam_name"].apply(is_cpt_code))
]["exam_name"].value_counts()
print(f"\nLevel-0 exam names (review to confirm correctly classified):")
print(unresolved.head(20))

In [ ]:
# Resolve multiple exams per ED stay
# If a patient had multiple imaging exams during one ED stay, keep the row
# with the highest acuity. Ties broken by earliest charttime — the first
# high-acuity exam ordered is the most clinically conservative choice.
df_rad = (
    df_rad
    .sort_values(["stay_id", "exam_acuity", "charttime"],
                 ascending=[True, False, True])
    .drop_duplicates(subset="stay_id", keep="first")
    .reset_index(drop=True)
)

In [ ]:
# Final table
rad_final = (
    df_rad[["subject_id", "stay_id", "hadm_id", "charttime", "exam_acuity"]]
    .rename(columns={
        "charttime":   "exam_time",
        "exam_acuity": "rad_acuity_level"
    })
    .copy()
)

print(f"\nShape: {rad_final.shape}")
print(rad_final["rad_acuity_level"].value_counts().sort_index())
print(rad_final.head())

rad_final.to_csv("radiology_features.csv", index=False)

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
from datasets import Dataset, DatasetInfo

PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

rad_info = DatasetInfo(
    description=(
        "Radiology imaging acuity labels derived from MIMIC-IV note.radiology and "
        "note.radiology_detail tables. "
        "One row per patient (subject_id). "
        "Filtered to exam_name field entries only; administrative and billing rows removed. "
        "field_value entries resolved across three formats (CPT code, exam code, literal name) "
        "via a CPT lookup table followed by string matching. "
        "rad_acuity_level reflects the highest imaging acuity exam ordered for the patient: "
        "0=no imaging or routine low-acuity exams only, 1=moderate acuity imaging "
        "(standard CT, MRI spine, ultrasound), 2=high-acuity imaging (CTA, CT head, "
        "MRI brain, portable chest, fluoroscopy/line placement). "
        "Patients absent from this table (no imaging after filtering) receive "
        "rad_acuity_level=0 at merge time."
    ),
    license=PHYSIONET_LICENSE,
)

ds_rad = Dataset.from_pandas(rad_final, split='radiology_features', info=rad_info)
ds_rad.push_to_hub("ADS599-Capstone/modeling_data", config_name="radiology_features", data_dir="modeling_data")
print("Pushed radiology_features to HuggingFace Hub.")